# Lab 2.3 &mdash; Sub-goals That Finish, and Knowing When to Re-plan

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 40 min &nbsp;|&nbsp; **Day 1 &middot; Module 2 &mdash; Agentic Planning &amp; Reasoning**

### What you'll do
- Have the model return a <code>Plan</code> object with declared dependencies
- Reject sub-goals no agent could ever call finished
- Tell a transient failure from a wrong plan &mdash; retry one, re-plan the other
- Run a plan against a tool that fails on purpose, and watch it recover

> **How this lab works.** You write real LangChain and LangGraph code. Fill every `BLANK`,
> then run the **Self-check** cell under each section &mdash; those check the *objects you built*
> (a bound tool, a compiled graph, an emitted tool call), so they are deterministic and do not
> depend on the model. Cells marked **Run it for real** put your code in front of the sandbox
> model; that is the part worth watching. The score line is feedback, not a grade.

> **Builds on Lab 2.2.** You have tools and a loop. This lab is about deciding what to
> do with them before you start, and what to do when the world disagrees.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-2-03")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError as exc:
        print(f"(a blank above is still unfilled: {exc} -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nSelf-check: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

# The served model reasons before it answers, and the reasoning is billed as completion
# tokens: 24.1s / 980 tokens with it on, 0.7s / 29 with it off, for the same answer. Off is
# the default here because you will make a lot of calls today. Pass think=True to see the
# difference for yourself -- and note that prompts written as an explicit ordered procedure
# survive thinking being off, while vague ones do not.
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm_cache = {}
def get_llm(temperature: float = 0.0, think: bool = False):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    from langchain_openai import ChatOpenAI
    key = (temperature, think)
    if key not in _llm_cache:
        kwargs = {} if think else {"extra_body": NO_THINK}
        _llm_cache[key] = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                                     api_key=LLM_API_KEY, temperature=temperature, **kwargs)
    return _llm_cache[key]

def ask(prompt: str, system: str | None = None, think: bool = False) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm(think=think).invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

def show_messages(messages, width: int = 88) -> None:
    """Print a message list the way a trace reads: type, content, and any tool calls."""
    for m in messages:
        kind = getattr(m, "type", "?")
        body = str(getattr(m, "content", "")).replace("\n", " ")[:width]
        calls = getattr(m, "tool_calls", None)
        line = f"  [{kind:9}] {body}"
        if calls:
            line += "  -> calls: " + ", ".join(f"{c['name']}({c['args']})" for c in calls)
        print(line)

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- the object-level self-checks still work)")

In [ ]:
# ------------------------------------------------- the case file (synthetic, self-contained)
# One domain runs through all five Module 2 labs: payment exceptions on a small ledger.
# Nothing here is real data and nothing leaves this notebook.

LEDGER = {
    "PMT-1001": {"amount": 250000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "settled",  "value_date": "2026-09-01", "reason_code": None},
    "PMT-1002": {"amount":  48250.75, "ccy": "EUR", "counterparty": "ACME-EU",
                 "status": "failed",   "value_date": "2026-09-02", "reason_code": "INSUFFICIENT_FUNDS"},
    "PMT-1003": {"amount": 990000.00, "ccy": "USD", "counterparty": "ZENITH",
                 "status": "held",     "value_date": "2026-09-02", "reason_code": "LIMIT_BREACH"},
    "PMT-1004": {"amount":   1200.00, "ccy": "GBP", "counterparty": "ACME-UK",
                 "status": "failed",   "value_date": "2026-09-03", "reason_code": "INVALID_IBAN"},
    "PMT-1005": {"amount": 750000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "held",     "value_date": "2026-09-03", "reason_code": "SANCTIONS_REVIEW"},
}

POLICY = {
    "INSUFFICIENT_FUNDS": "Retry once after 24h. If it fails again, notify the client desk. No manual funding.",
    "LIMIT_BREACH":       "Payments above USD 500,000 need Treasury approval before release.",
    "INVALID_IBAN":       "Return to originator with code R04. Never repair beneficiary details in-house.",
    "SANCTIONS_REVIEW":   "Hold. Compliance decides. Operations must not release or cancel.",
}

# Which reason codes may an agent resolve on its own, and which need a human?
NEEDS_HUMAN = {"LIMIT_BREACH", "SANCTIONS_REVIEW"}

print(f"{len(LEDGER)} payments, {len(POLICY)} policy rules loaded")

In [ ]:
# ------------------------------------------------- the two tools, carried through Module 2
from langchain_core.tools import tool

@tool
def lookup_payment(ref: str) -> str:
    """Return the ledger record for one payment reference such as 'PMT-1003'.

    Use when you need the status, amount, counterparty or reason code of a specific payment.
    """
    rec = LEDGER.get(ref)
    return json.dumps({"ref": ref, **rec}) if rec else f"no payment found with reference {ref!r}"


@tool
def policy_for(reason_code: str) -> str:
    """Return the operating policy for one failure reason code such as 'LIMIT_BREACH'.

    Use after you know why a payment failed and need to know what to do about it.
    """
    return POLICY.get(reason_code, f"no policy on file for reason code {reason_code!r}")


TOOLS = {t.name: t for t in (lookup_payment, policy_for)}
print("tools:", list(TOOLS))

## Concept

An agent that plans badly fails in one of two ways, and they need opposite responses:

- the **world** misbehaved &mdash; a timeout, a rate limit, a blip. Retry.
- the **plan** was wrong &mdash; the tool does not exist, the argument is invalid, the step depends
  on something that never happened. Retrying is a loop that burns budget and finishes nowhere.

Getting that distinction wrong is one of the commonest production failures in agents, and it is
almost always the same bug: a blanket `except: retry`.

## Section 1 &mdash; A plan the model returns as an object

`with_structured_output(Plan)` sends the schema to the model and validates what comes back, so
you get a `Plan`, not a paragraph about a plan. Fill in the field descriptions &mdash; they are the
only instructions the model gets about what each field means.

In [ ]:
from typing import List, Literal
from pydantic import BaseModel, Field

class Step(BaseModel):
    """One step of an investigation plan."""
    name: str = Field(description="Short snake_case name for this step")
    tool: str = Field(description="BLANK")          # TODO: how does the model know what may go here?
    argument: str = Field(description="The single argument to pass to the tool")
    depends_on: List[str] = Field(default_factory=list,
                                  description="Names of steps that must finish before this one")


class Plan(BaseModel):
    """An investigation plan for one payment exception."""
    goal: str = Field(description="The question this plan answers, in one line")
    steps: List[Step] = Field(description="The steps, which may be listed in any order")


def plan_for(goal: str) -> Plan:
    """Ask the model for a Plan object."""
    return get_llm().with_structured_output(Plan).invoke(
        "Produce an investigation plan for this goal. Every step must name a tool from the list "
        "and declare the steps it depends on.\n"
        f"TOOLS: {list(TOOLS)}\nGOAL: {goal}")

### A sub-goal an agent cannot finish

"Understand the payment thoroughly" has no completion test. An agent handed it will either stop
arbitrarily or never stop. Reject those before you run them.

In [ ]:
VAGUE = ("understand", "thoroughly", "make sure", "as needed", "properly",
         "investigate fully", "look into", "analyse", "review", "consider")

def finishable(step: Step) -> bool:
    """A step is finishable when it names a real tool and does not describe an attitude."""
    if step.tool != "none" and step.tool not in TOOLS:
        return False
    return not any(word in step.name.lower().replace("_", " ") for word in VAGUE)

In [ ]:
# --- Self-check: Section 1   (Pydantic objects only -- no model call)
_ok    = Step(name="read_payment", tool="lookup_payment", argument="PMT-1003")
_vague = Step(name="understand_the_case", tool="none", argument="")
_ghost = Step(name="email_the_desk", tool="send_email", argument="ops@example.com")

def _tool_desc():
    d = Step.model_fields["tool"].description
    if not d or d == "BLANK":
        raise NameError("Step.tool still has no description")
    return d

check("the plan schema declares a goal and steps",
      lambda: set(Plan.model_fields) == {"goal", "steps"})
check("Step.tool tells the model where the valid names come from",
      lambda: len(_tool_desc()) > 40 and "none" in _tool_desc(),
      "the model cannot guess an enum it was never shown -- name the source and the escape hatch")
check("a concrete step is finishable",   lambda: finishable(_ok) is True)
check("a vague step is rejected",        lambda: finishable(_vague) is False,
      '"understand the case" has no completion test, so an agent cannot stop')
check("a step naming a tool that does not exist is rejected",
      lambda: finishable(_ghost) is False,
      "this is a WRONG PLAN, not a transient failure -- Section 2 depends on the difference")
check("a reasoning step with tool='none' is allowed",
      lambda: finishable(Step(name="decide_action", tool="none", argument="")) is True)

## Section 2 &mdash; Transient, or wrong?

Look at what came back and decide which kind of failure it is. Retry the world; re-plan the plan.

In [ ]:
TRANSIENT = ("timeout", "timed out", "503", "502", "429", "connection reset",
             "temporarily unavailable", "rate limit", "try again")

PERMANENT = ("no such tool", "no payment found", "invalid", "not permitted",
             "no policy on file", "unknown reason code")

def diagnose(observation: str) -> Literal["transient", "wrong_plan", "ok"]:
    """What kind of thing just happened?"""
    low = (observation or "").lower()
    if any(w in low for w in TRANSIENT):
        return "transient"
    if any(w in low for w in BLANK):  # TODO: retrying will never fix these -- which list is it?
        return "wrong_plan"
    return "ok"


def response_to(kind: str, attempts: int, max_retries: int = 2) -> str:
    """What should the agent do about it?"""
    if kind == "ok":
        return "continue"
    if kind == "transient":
        return "retry" if attempts < max_retries else "give_up"
    return "replan"

In [ ]:
# --- Self-check: Section 2
check("a timeout is transient",        lambda: diagnose("upstream timed out after 30s") == "transient")
check("a 429 is transient",            lambda: diagnose("HTTP 429 rate limit") == "transient")
check("a missing record is a wrong plan",
      lambda: diagnose("no payment found with reference 'PMT-9999'") == "wrong_plan",
      "retrying this forever is the classic burn -- the reference will not appear")
check("an unknown tool is a wrong plan", lambda: diagnose("no such tool 'send_email'") == "wrong_plan")
check("a good observation is ok",
      lambda: diagnose('{"ref": "PMT-1003", "status": "held"}') == "ok")
check("a transient failure is retried, then abandoned",
      lambda: [response_to("transient", i) for i in range(4)]
              == ["retry", "retry", "give_up", "give_up"])
check("a wrong plan is never retried",
      lambda: all(response_to("wrong_plan", i) == "replan" for i in range(5)),
      "retrying a wrong plan is a loop that finishes nowhere")

## Section 3 &mdash; Run the plan, and recover

`execute()` walks the ordered steps, runs each tool, and reacts to what comes back. The flaky
wrapper below fails the first *n* calls with a timeout, so you can watch both paths without
waiting for a real outage.

In [ ]:
def order_steps(plan: Plan) -> list[Step]:
    """Dependency order. Raises ValueError on a cycle or a missing dependency."""
    by_name = {s.name: s for s in plan.steps}
    ordered, done = [], set()
    while len(ordered) < len(by_name):
        progressed = False
        for name, step in by_name.items():
            if name in done or not all(d in done for d in step.depends_on):
                continue
            ordered.append(step); done.add(name); progressed = True
        if not progressed:
            raise ValueError("cycle or missing dependency in plan")
    return ordered


def flaky(fail_times: int):
    """A tool runner that fails with a timeout the first `fail_times` calls."""
    state = {"n": 0}
    def run(step: Step) -> str:
        state["n"] += 1
        if state["n"] <= fail_times:
            return "upstream timed out after 30s"
        tool_obj = TOOLS.get(step.tool)
        if tool_obj is None:
            return f"no such tool {step.tool!r}"
        arg = list(tool_obj.args)[0]
        return str(tool_obj.invoke({arg: step.argument}))
    return run


def execute(plan: Plan, run, max_retries: int = 2) -> dict:
    """Run the plan. Retry transient failures; stop and report a wrong plan."""
    trace, attempts = [], 0
    for step in order_steps(plan):
        attempts = 0
        while True:
            observation = run(step)
            kind = diagnose(observation)
            action = response_to(kind, attempts, max_retries)
            trace.append((step.name, kind, action))
            if action == "continue":
                break
            if action == "retry":
                attempts += 1
                continue
            return {"trace": trace, "outcome": BLANK, "failed_at": step.name}  # TODO: what ended it?
    return {"trace": trace, "outcome": "completed", "failed_at": None}

In [ ]:
# --- Self-check: Section 3   (a hand-written plan and a fake runner -- no model call)
HAND_PLAN = Plan(goal="Decide what to do about PMT-1003", steps=[
    Step(name="read_policy",  tool="policy_for",     argument="LIMIT_BREACH",
         depends_on=["read_payment"]),
    Step(name="read_payment", tool="lookup_payment", argument="PMT-1003"),
])
BAD_PLAN = Plan(goal="g", steps=[Step(name="email_desk", tool="send_email", argument="x")])

check("dependencies are ordered first",
      lambda: [s.name for s in order_steps(HAND_PLAN)] == ["read_payment", "read_policy"])
check("a clean run completes",
      lambda: execute(HAND_PLAN, flaky(0))["outcome"] == "completed")
check("one timeout is retried and then succeeds",
      lambda: execute(HAND_PLAN, flaky(1))["outcome"] == "completed")
check("the retry is visible in the trace",
      lambda: any(a == "retry" for _, _, a in execute(HAND_PLAN, flaky(1))["trace"]))
check("endless timeouts are eventually abandoned",
      lambda: execute(HAND_PLAN, flaky(99))["outcome"] == "give_up",
      "max_retries has to be a number, not a hope")
check("a wrong plan triggers a replan, not a retry",
      lambda: execute(BAD_PLAN, flaky(0))["outcome"] == "replan")
check("a wrong plan is caught on the FIRST attempt",
      lambda: len(execute(BAD_PLAN, flaky(0))["trace"]) == 1,
      "no point calling a tool that does not exist twice")

## Run it for real

The model plans; your code decides whether the plan is runnable; then it runs against a tool that
fails twice before it works.

In [ ]:
if llm_ready():
    def _plan_and_run():
        plan = plan_for("Find out why PMT-1003 is held and what policy requires.")
        print("goal:", plan.goal)
        for s in plan.steps:
            flag = "ok  " if finishable(s) else "DROP"
            print(f"  [{flag}] {s.name:24} tool={s.tool:16} arg={s.argument:16} after={s.depends_on}")

        runnable = Plan(goal=plan.goal, steps=[s for s in plan.steps if finishable(s)])
        if not runnable.steps:
            print("\nnothing runnable in that plan -- which is itself a result")
            return None

        print("\n--- executing against a tool that times out twice ---")
        result = execute(runnable, flaky(2))
        for name, kind, action in result["trace"]:
            print(f"  {name:24} {kind:12} -> {action}")
        print(f"\noutcome: {result['outcome']}")
        return result
    RESULT = guard(_plan_and_run)

### Read it

Look at the plan the model produced *before* anything ran. Some of the time it will invent a step
naming a tool that does not exist &mdash; `check_sanctions`, `notify_desk`, something plausible. That
is not the model being bad; it is the model doing what planners do. `finishable()` catching it
before execution is the entire value of validating a plan.

Then look at the trace. Two timeouts, two retries, then progress &mdash; and the retries cost you
nothing but time. Had `diagnose` mislabelled the missing-tool case as transient, you would have
watched it retry a step that can never succeed until the budget ran out. That single misjudgement
is the bug this lab exists to prevent, and in real code it always looks like:

```python
except Exception:
    retry()          # what kind of exception? nobody asked
```

In [ ]:
score()

## Your turn

1. `execute` gives up on a wrong plan. Make it actually **re-plan**: feed the failed step and its
   observation back to `plan_for` and run the new plan. Cap the number of re-plans, and say why
   your cap is the right one.
2. Add a third diagnosis, `needs_human`, for observations that name a sanctions hold. What should
   `response_to` return for it, and why is that different from `give_up`?
3. `finishable()` uses a word list, which is crude. Replace it with a small
   `with_structured_output` call that asks the model whether a step has a completion test. Run
   both over ten invented steps and see which you trust more.